In [ ]:
import os
import logging
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import re
from sentence_transformers import SentenceTransformer
import chromadb
from sklearn.manifold import TSNE
from litellm import completion
from tqdm import tqdm
from agents.items import Item


In [ ]:
load_dotenv(override=True)
DB="products_vectorstore"

In [ ]:
hf_token = os.environ['HF_token']
login(token=hf_token,add_to_git_credential=True)

In [ ]:
LITE_MODE=True
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")


In [ ]:
client = chromadb.PersistentClient(path=DB)
# will create the folder and client object

In [ ]:
#now we need to encode we will We will be using a free encoder model in the huggin face known as all-mini-lm l6v2. 
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
vector = encoder.encode(['A ai proficient engineer about to complete the lessons of uman society'])
vector.shape

In [ ]:
vector

In [ ]:
collection_name = "products"
existing_collection_names = [collection.name for collection in client.list_collections()]
if collection_name not in existing_collection_names:
    collection = client.create_collection(collection_name)
    for i in tqdm(range(0,len(train),1000)):
        end_index = min(i + 1000, len(train))
        documents = [train[j].summary for j in range(i,end_index)]
        vectors  = encoder.encode(documents).astype(float).tolist()
        metadata = [{'category':train[j].category,'price':train[j].price} for j in range(i,end_index)]
        ids = [f"doc_{j}" for j in range(i, end_index)]
        ids=ids[:len(documents)]
        collection.add(ids=ids, documents=documents, embeddings=vectors, metadatas=metadata)
collection = client.get_or_create_collection(collection_name)
        

In [ ]:
CATEGORIES = ['Appliances', 'Automotive', 'Cell_Phones_and_Accessories', 'Electronics','Musical_Instruments', 'Office_Products', 'Tools_and_Home_Improvement', 'Toys_and_Games']
COLORS = ['cyan', 'blue', 'brown', 'orange', 'yellow', 'green' , 'purple', 'red']

In [ ]:
# Prework
result = collection.get(include=['embeddings', 'documents', 'metadatas'], limit=10000)
vectors = np.array(result['embeddings'])
documents = result['documents']
categories = [metadata['category'] for metadata in result['metadatas']]
colors = [COLORS[CATEGORIES.index(c)] for c in categories]

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2,random_state=42)
reduced_vectors=tsne.fit_transform(vectors)
import plotly.express as ex

hover_text =[
  f"Text:{doc[:20]}" for doc in (documents)
]

fig_2d = ex.scatter(
  x=reduced_vectors[:,0],
  y=reduced_vectors[:,1],
  color=categories,
  hover_data={
    "text":hover_text
  }
)


In [ ]:
fig_2d.show()

In [ ]:
from sklearn.manifold import TSNE

import plotly.express as ex

tsne = TSNE(n_components=3)
reduced_vectors = tsne.fit_transform(vectors)

fig_3d = ex.scatter_3d(
    x=reduced_vectors[:,0],
    y=reduced_vectors[:,1],
    z=reduced_vectors[:,2],
    color=categories,
    hover_data={"text":hover_text}
)

fig_3d.show()

In [ ]:
def vector(item):
    return encoder.encode(item.summary)
# vector(test[0])

In [ ]:
def find_similars(item):
    vec = vector(item)
    results = collection.query(query_embeddings=vec.astype('float').tolist(),n_results=5)
    documents = results['documents'][0][:]
    prices = [m['price'] for m in results['metadatas'][0][:]]
    return documents,prices

find_similars(test[0])

In [ ]:
# We need to give some context to GPT-5.1 by selecting 5 products with similar descriptions

def make_context(similars, prices):
    message = "For context, here are some other items that might be similar to the item you need to estimate.\n\n"
    for similar, price in zip(similars, prices):
        message += f"Potentially related product:\n{similar}\nPrice is ${price:.2f}\n\n"
    return message

In [ ]:
documents, prices = find_similars(test[0])
print(make_context(documents, prices))

In [ ]:
def messages_for(item, similars, prices):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}\n\n"
    message += make_context(similars, prices)
    return [{"role": "user", "content": message}]

In [ ]:
documents, prices = find_similars(test[0])
print(messages_for(test[0], documents, prices)[0]['content'])

In [ ]:
def gpt_5_1_rag(item):
    documents,price = find_similars(item)
    response = completion(model="gpt-5.1",messages = messages_for(item=item,similars=documents,prices=price))
    return response.choices[0].message.content
gpt_5_1_rag(test[0])

In [ ]:
test[0].price

In [ ]:
from agents.evaluator import evaluate
evaluate(gpt_5_1_rag, test)

In [ ]:
import modal 
Pricer = modal.Cls.from_name("pricer-service","Pricer")
pricer = Pricer()

In [ ]:
from agents.fronteir_agent import FronteirAgent

agent = FronteirAgent(collection)
agent.price("Whey protein ATOM 27g")